# Step 4 — Building reusable patient features

In the previous notebook we manually processed one patient at a time. Here we turn that logic into a reusable Python function.

The core idea is:

**patient file → first 6 observations → feature summary → future target**

For now, this is still a learning version. The exact clinical prediction horizon will be refined later.

In [ ]:
import pandas as pd

base_url = "https://physionet.org/files/challenge-2019/1.0.0/training/training_setA/"

## 1. What is a function?

A function is a reusable block of code. Instead of repeating the same steps for every patient, we define the procedure once and then call it with a patient ID.

In Python, functions start with `def`.

In [ ]:
def load_patient(patient_id):
    url = base_url + patient_id + ".psv"
    patient = pd.read_csv(url, sep="|")
    return patient

This function has three steps:

1. it receives `patient_id`;
2. it builds the PhysioNet URL and reads the file;
3. `return patient` gives the loaded DataFrame back to us.

In [ ]:
patient = load_patient("p000001")
patient.head()

## 2. Build one patient feature row

Now we create a second function that turns the raw time series into one row of features.

Important rules in this first version:
- use only the first 6 observations to create predictive features;
- require at least one later observation, otherwise there is no future to predict;
- exclude patients whose `SepsisLabel` is already positive inside the first 6 observations;
- handle missing HR values before computing the simple HR change.

In [ ]:
def build_patient_features(patient_id):
    patient = load_patient(patient_id)

    # We need at least 6 observations for the initial window
    # plus at least 1 later observation to define a future target.
    if len(patient) <= 6:
        return None

    first_6h = patient.head(6)
    future = patient.iloc[6:]

    # If the warning label is already positive in the first window,
    # this patient is not suitable for this prediction setup.
    if first_6h["SepsisLabel"].max() == 1:
        return None

    valid_hr = first_6h["HR"].dropna()

    if len(valid_hr) >= 2:
        hr_change = valid_hr.iloc[-1] - valid_hr.iloc[0]
    else:
        hr_change = float("nan")

    features = {
        "patient": patient_id,
        "mean_HR_6h": first_6h["HR"].mean(),
        "min_HR_6h": first_6h["HR"].min(),
        "max_HR_6h": first_6h["HR"].max(),
        "std_HR_6h": first_6h["HR"].std(),
        "change_HR_6h": hr_change,
        "min_O2Sat_6h": first_6h["O2Sat"].min(),
        "mean_Resp_6h": first_6h["Resp"].mean(),
        "mean_MAP_6h": first_6h["MAP"].mean(),
        "future_sepsis": future["SepsisLabel"].max()
    }

    return features

### Why can a function return `None`?

`None` means that this patient is not usable under our current rules.

Examples:
- too few observations;
- already positive during the first 6 observations.

This is cleaner than forcing every patient into the dataset.

## 3. Test the function on one patient

In [ ]:
result = build_patient_features("p000001")
result

If the patient satisfies our rules, the output is one dictionary: one future dataset row.

If the output is `None`, that patient was excluded by the current rules.

## 4. A useful counterexample: p000034

Earlier we saw that this patient receives a positive sepsis warning label very early. Under the current setup it may therefore be excluded.

In [ ]:
build_patient_features("p000034")

## 5. Apply the same function to multiple patients

Now the advantage of the function becomes obvious: we can process many patients without copying all the feature-engineering code each time.

In [ ]:
dataset_rows = []

for i in range(1, 101):
    patient_id = f"p{i:06d}"
    row = build_patient_features(patient_id)

    if row is not None:
        dataset_rows.append(row)

dataset = pd.DataFrame(dataset_rows)
dataset

The new object `dataset` has a different meaning from the raw PhysioNet files:

- **one row = one eligible patient**;
- columns such as `mean_HR_6h` and `min_O2Sat_6h` are **features**;
- `future_sepsis` is the **target**.

In [ ]:
print("Eligible patients:", len(dataset))
dataset["future_sepsis"].value_counts(dropna=False)

## 6. Check missing values in the engineered dataset

Even after summarizing patients, some features can still be missing because the underlying measurements were not recorded.

In [ ]:
dataset.isna().sum().sort_values(ascending=False)

## Current checkpoint

At this point we have learned how to:
- define a Python function with `def`;
- use input arguments such as `patient_id`;
- use `return` to send a result back;
- exclude unusable patients with `None`;
- convert raw longitudinal patient data into one feature row;
- apply the same transformation to many patients.

### Important limitation

Our target is still **any later positive sepsis label during the remaining stay**. That is too broad for the final project. The next methodological step is to define a fixed future prediction window, for example the following 6 hours, before training a model.